<a href="https://colab.research.google.com/github/Chuxian-Chen/Master-Experiment/blob/main/Fair_aware_Loan_approval_decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from interpret.glassbox import ExplainableBoostingClassifier
from pygam import LogisticGAM, f as gam_f, l as gam_l
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tabpfn import TabPFNClassifier
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
print("\n1. Loading and Preprocessing Data...")
file_name = '../../DataSet/Loan Approval Insights Data to Decision-Making.csv'
if not os.path.exists(file_name):
    print(f"Error: File not found: {file_name}");
    exit()

df = pd.read_csv(file_name)
le_dict = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

try:
    doc_id = le_dict['person_education'].transform(['Doctorate'])[0]
except:
    doc_id = 1

S = np.where(df['person_education'] == doc_id, 1, 0)
X = df.drop(columns=['loan_status'])
y = df['loan_status']
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


1. Loading and Preprocessing Data...


In [ ]:
def train_and_evaluate(model_name, model_instance, X, y, S, seed, mode='detailed', is_pre_trained=False,
                       saved_bounds=None):
    X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
        X, y, S, test_size=0.2, random_state=seed, stratify=y
    )

    current_model_bounds = None

    if not is_pre_trained:
        if model_name == "TabPFN":
            _, X_train_sub, _, y_train_sub = train_test_split(
                X_train, y_train, test_size=min(1000, len(X_train)), random_state=seed, stratify=y_train
            )
            model_instance.fit(X_train_sub, y_train_sub)
        elif model_name == "GAM":
            # Limit sampling to prevent memory overflow
            sample_size = min(3000, len(X_train))
            _, X_train_sub, _, y_train_sub = train_test_split(
                X_train, y_train, test_size=sample_size, random_state=seed, stratify=y_train
            )
            model_instance = LogisticGAM(
                gam_f(0) + gam_f(1) + gam_f(2) + gam_f(3) + gam_f(4) + gam_f(5) + gam_f(6) + gam_f(7) +
                gam_l(8) + gam_l(9) + gam_l(10) + gam_l(11)
            ).fit(X_train_sub, y_train_sub)
            current_model_bounds = (X_train_sub.min().to_numpy(), X_train_sub.max().to_numpy())
        else:
            if hasattr(model_instance, 'random_state'):
                model_instance.set_params(random_state=seed)
            model_instance.fit(X_train, y_train)
    else:
        if model_name == "GAM":
            if saved_bounds is not None:
                current_model_bounds = saved_bounds
            else:
                current_model_bounds = (X.min().to_numpy(), X.max().to_numpy())

    if model_name == "GAM":
        X_test_arr = X_test.to_numpy()
        X_test_clipped = np.clip(X_test_arr, current_model_bounds[0], current_model_bounds[1])
        y_prob = model_instance.predict_mu(X_test_clipped)
    else:
        y_prob = model_instance.predict_proba(X_test)[:, 1]

    y_pred_base = (y_prob >= 0.5).astype(int)
    acc_base = accuracy_score(y_test, y_pred_base)
    abs_spd_base = abs(y_pred_base[S_test == 1].mean() - y_pred_base[S_test == 0].mean())

    # Fairness: Threshold-based shifting strategy
    candidates = []
    thresholds = np.linspace(0.4, 0.6, 21)
    for th_doc in thresholds:
        for th_rest in thresholds:
            preds = np.array([1 if (p >= th_doc if s == 1 else p >= th_rest) else 0 for p, s in zip(y_prob, S_test)])
            curr_spd = abs(preds[S_test == 1].mean() - preds[S_test == 0].mean())
            acc = accuracy_score(y_test, preds)
            if curr_spd < abs_spd_base and acc > (acc_base * 0.95):
                candidates.append({'spd': curr_spd, 'acc': acc, 'preds': preds, 'th_doc': th_doc, 'th_rest': th_rest})

    if candidates:
        candidates.sort(key=lambda x: x['acc'], reverse=True)
        best = candidates[0]
        res = {
            'Model': model_name, 'Instance': model_instance, 'Bounds': current_model_bounds,
            'Base_Acc': acc_base, 'Base_Prec': precision_score(y_test, y_pred_base, zero_division=0),
            'Base_Rec': recall_score(y_test, y_pred_base, zero_division=0),
            'Base_F1': f1_score(y_test, y_pred_base, zero_division=0),
            'Base_SPD': abs_spd_base,
            'Fair_Acc': best['acc'], 'Fair_Prec': precision_score(y_test, best['preds'], zero_division=0),
            'Fair_Rec': recall_score(y_test, best['preds'], zero_division=0),
            'Fair_F1': f1_score(y_test, best['preds'], zero_division=0),
            'Fair_SPD': best['spd'], 'Best_Th_Doc': best['th_doc'], 'Best_Th_Rest': best['th_rest']
        }
    else:
        res = {
            'Model': model_name, 'Instance': model_instance, 'Bounds': current_model_bounds,
            'Base_Acc': acc_base, 'Base_Prec': precision_score(y_test, y_pred_base, zero_division=0),
            'Base_Rec': recall_score(y_test, y_pred_base, zero_division=0),
            'Base_F1': f1_score(y_test, y_pred_base, zero_division=0),
            'Base_SPD': abs_spd_base,
            'Fair_Acc': acc_base, 'Fair_SPD': abs_spd_base, 'Best_Th_Doc': 0.5, 'Best_Th_Rest': 0.5
        }
    return res if mode == 'detailed' else (abs_spd_base, res.get('Fair_SPD', abs_spd_base))


def calculate_paired_t_test_metrics(baseline_scores, fair_scores):
    x, y_vals = np.array(baseline_scores), np.array(fair_scores)
    n = len(x);
    d = x - y_vals
    d_bar, S_d = np.mean(d), np.std(d, ddof=1)
    se = S_d / np.sqrt(n)
    t0 = d_bar / se if se != 0 else 0
    p_val = stats.t.sf(np.abs(t0), n - 1) * 2
    return d_bar, S_d, t0, p_val

In [ ]:
save_dir = '../Saved_Results'
if not os.path.exists(save_dir): os.makedirs(save_dir)
model_package_path = os.path.join(save_dir, 'loan_model_package.joblib')

is_pre_trained = False
trained_instances = {}
saved_gam_bounds = None

if os.path.exists(model_package_path):
    print(f"\nFound saved model package. Start loading...")
    package = joblib.load(model_package_path)
    trained_instances = package['models']
    saved_gam_bounds = package.get('gam_bounds')
    models_to_run = trained_instances
    is_pre_trained = True
else:
    print("\nNo model found. Start training...")
    models_to_run = {
        "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(min_samples_leaf=50, random_state=42),
        "EBM": ExplainableBoostingClassifier(random_state=42),
        "GAM": "LogisticGAM",
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "TabPFN": TabPFNClassifier(device='cpu')
    }

# --- Table 4-6 ---
print("\nPreparing Table 4-6...")
detailed_results, final_thresholds = [], {}
for name, clf in models_to_run.items():
    print(f"    Processing {name}...")
    instance = clf if not (name == "GAM" and isinstance(clf, str)) else None
    m = train_and_evaluate(name, instance, X_scaled, y, S, seed=42, mode='detailed',
                           is_pre_trained=is_pre_trained, saved_bounds=saved_gam_bounds)

    detailed_results.append({
        'Algorithm': name, 'Strategy': 'Baseline', 'Accuracy': m['Base_Acc'],
        'Precision': m['Base_Prec'], 'Recall': m['Base_Rec'], 'F1 Score': m['Base_F1'], 'SPD (Bias)': m['Base_SPD']
    })
    detailed_results.append({
        'Algorithm': name, 'Strategy': 'Fairness-Aware', 'Accuracy': m['Fair_Acc'],
        'Precision': m.get('Fair_Prec', m['Base_Prec']), 'Recall': m.get('Fair_Rec', m['Base_Rec']),
        'F1 Score': m.get('Fair_F1', m['Base_F1']), 'SPD (Bias)': m['Fair_SPD']
    })
    final_thresholds[name] = (m['Best_Th_Doc'], m['Best_Th_Rest'])
    if not is_pre_trained:
        trained_instances[name] = m['Instance']
        if name == "GAM": saved_gam_bounds = m['Bounds']

df_4_6 = pd.DataFrame(detailed_results).round(4)
print("\n[Table 4-6: Model Performance Comparison]")
print(df_4_6.to_string(index=False))

if not is_pre_trained:
    print("\nSaving package...")
    joblib.dump({'models': trained_instances, 'scaler': scaler, 'le_dict': le_dict,
                 'thresholds': final_thresholds, 'gam_bounds': saved_gam_bounds, 'features': X.columns.tolist()},
                model_package_path)


Found saved model package. Start loading...

Preparing Table 4-6...
    Processing Logistic Regression...
    Processing Decision Tree...
    Processing EBM...
    Processing GAM...
    Processing Random Forest...
    Processing XGBoost...
    Processing TabPFN...

[Table 4-6: Model Performance Comparison]
          Algorithm       Strategy  Accuracy  Precision  Recall  F1 Score  SPD (Bias)
Logistic Regression       Baseline    0.8968     0.7779  0.7495    0.7634      0.0252
Logistic Regression Fairness-Aware    0.8982     0.7898  0.7385    0.7633      0.0177
      Decision Tree       Baseline    0.9190     0.8513  0.7700    0.8086      0.0245
      Decision Tree Fairness-Aware    0.9194     0.8505  0.7735    0.8102      0.0143
                EBM       Baseline    0.9346     0.8961  0.7980    0.8442      0.0362
                EBM Fairness-Aware    0.9350     0.8741  0.8265    0.8497      0.0062
                GAM       Baseline    0.8544     0.7322  0.5440    0.6242      0.0486
   

In [ ]:
# --- Table 4-7 & 4-8 ---
print("\nPreparing Table 4-7 & 4-8 (10 Iterations)...")
raw_rows, summary_rows = [], []
for name, clf in trained_instances.items():
    print(f"    Validating {name}...")
    b_spds, fair_scores_list = [], []
    for i in range(10):
        b_val, fair_spd = train_and_evaluate(name, clf, X_scaled, y, S, seed=i, mode='spd_only',
                                             is_pre_trained=True, saved_bounds=saved_gam_bounds)
        b_spds.append(b_val);
        fair_scores_list.append(fair_spd)

    raw_rows.append([name] + b_spds);
    raw_rows.append([f"Fair-{name}"] + fair_scores_list)
    d_bar, S_d, t_stat, p_val = calculate_paired_t_test_metrics(b_spds, fair_scores_list)
    summary_rows.append({
        'Model': name, 'Baseline Mean SPD': np.mean(b_spds), 'Fair Model Mean SPD': np.mean(fair_scores_list),
        'Bias Reduction': f"{((np.mean(b_spds) - np.mean(fair_scores_list)) / np.mean(b_spds)) * 100:.1f}%",
        'd_bar': d_bar, 'S_d': S_d, 't-stat': t_stat, 'P-Value': p_val, 'Sig?': 'Yes' if p_val < 0.05 else 'No'
    })

df_4_7 = pd.DataFrame(raw_rows, columns=['Model'] + [f'Run {i + 1}' for i in range(10)]).round(4)
df_4_8 = pd.DataFrame(summary_rows).round(4)
print("\n[Table 4-7]\n", df_4_7.to_string(index=False))
print("\n[Table 4-8]\n", df_4_8.to_string(index=False))

# --- Save evaluation results ---
df_4_6.to_csv(os.path.join(save_dir, 'Table_4_6_Performance.csv'), index=False)
df_4_7.to_csv(os.path.join(save_dir, 'Table_4_7_SPD_Runs.csv'), index=False)
df_4_8.to_csv(os.path.join(save_dir, 'Table_4_8_Significance.csv'), index=False)
print(f"\nEvaluation results have been saved as CSV files in '{save_dir}/'")


Preparing Table 4-7 & 4-8 (10 Iterations)...
    Validating Logistic Regression...
    Validating Decision Tree...
    Validating EBM...
    Validating GAM...
    Validating Random Forest...
    Validating XGBoost...
    Validating TabPFN...

[Table 4-7]
                    Model  Run 1  Run 2  Run 3  Run 4  Run 5  Run 6  Run 7  Run 8  Run 9  Run 10
     Logistic Regression 0.0465 0.0160 0.0127 0.0374 0.0312 0.0152 0.0364 0.0082 0.0435  0.0218
Fair-Logistic Regression 0.0430 0.0070 0.0005 0.0254 0.0241 0.0086 0.0286 0.0070 0.0107  0.0184
           Decision Tree 0.0660 0.0050 0.0272 0.0015 0.0373 0.0084 0.0553 0.0029 0.0112  0.0001
      Fair-Decision Tree 0.0358 0.0001 0.0066 0.0004 0.0044 0.0027 0.0317 0.0019 0.0066  0.0001
                     EBM 0.0365 0.0100 0.0117 0.0350 0.0209 0.0063 0.0484 0.0137 0.0027  0.0258
                Fair-EBM 0.0194 0.0020 0.0098 0.0145 0.0204 0.0039 0.0473 0.0003 0.0022  0.0233
                     GAM 0.0342 0.0404 0.0013 0.0449 0.0048 0.0394 0.01